In [0]:
import dlt
from pyspark.sql.functions import col, current_timestamp, year, month, dayofmonth, hour, hash, abs

In [0]:
silver_schema_name = spark.conf.get("pipeline.silver_schema_name", "weather_silver")
gold_schema_name = spark.conf.get("pipeline.gold_schema_name", "weather_gold")

In [0]:
@dlt.table(
    name=f"{gold_schema_name}.dim_city",
    comment="Dimension table that holds all the city names",
    table_properties={"quality": "gold"}
)
def dim_city():
    weather_cities = dlt.read(f"{silver_schema_name}.unified_weather").select("city", "lat", "lon")
    rainfall_cities = dlt.read(f"{silver_schema_name}.unified_rainfall").select("city", "lat", "lon")
    
    return (
        weather_cities.union(rainfall_cities)
        .distinct()
        .withColumn("city_id", abs(hash(col("city"))))                     # hash in order to make the name unique
    )

In [0]:
@dlt.table(
    name=f"{gold_schema_name}.dim_date",
    comment="Dimension table that holds all the dates",
    table_properties={"quality": "gold"}
)
def dim_date():
    weather_times = dlt.read(f"{silver_schema_name}.unified_weather").select(
        col("event_time").alias("date_id")
    )
    rainfall_times = dlt.read(f"{silver_schema_name}.unified_rainfall").select(
        col("event_time").alias("date_id")
    )
    
    return (
        weather_times.union(rainfall_times)
        .distinct()
        .withColumn("year", year(col("date_id")))
        .withColumn("month", month(col("date_id")))
        .withColumn("day", dayofmonth(col("date_id")))
        .withColumn("hour", hour(col("date_id")))
    )

In [0]:
@dlt.table(
    name=f"{gold_schema_name}.fact_weather",
    comment="Fact table for weather indicators",
    table_properties={"quality": "gold"}
)
def fact_weather_ind():   
    
    return (
        dlt.read_stream(f"{silver_schema_name}.unified_weather") 
        .select(
        abs(hash(col("city"))).alias("city_id"),
        col("event_time").alias("date_id"),  
        col("temperature"),                         
        col("pm2_5"),
        col("pm10"),
        col("wind_speed"),
        col("wind_direction"),
        col("air_quality")       
        )          
    )

In [0]:
@dlt.table(
    name=f"{gold_schema_name}.fact_rainfall",
    comment="Fact table for rainfall",
    table_properties={"quality": "gold"}
)
def fact_rainfall():   
    
    return (
        dlt.read_stream(f"{silver_schema_name}.unified_rainfall") 
        .select(
            abs(hash(col("city"))).alias("city_id"),
            col("event_time").alias("date_id"),           
            col("rain_sum"),
            col("showers_sum"),
            col("snowfall_sum")            
        )
    )